Dependencies

In [ ]:
import numpy as np
import pandas as pd
import math
import os
import json
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

Configuration

In [ ]:
def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "artifacts").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(f"Could not locate the MIRx project root from {start}")

PROJECT_ROOT = find_project_root()
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
GROUND_TRUTH_DIR = ARTIFACTS_DIR / "evaluation" / "ground_truth.json"
TEST_QUERIES_DIR = ARTIFACTS_DIR / "evaluation" / "test_queries.json"

TOP_K_VALUES = [1, 3, 5, 10]

Load data

In [ ]:
embeddings = np.load(os.path.join(ARTIFACTS_DIR, "embeddings.npy"))
metadata = pd.read_parquet(os.path.join(ARTIFACTS_DIR, "metadata.parquet"))
if "document_id" not in metadata.columns:
    metadata = metadata.copy()
    metadata["document_id"] = "doc_" + pd.Series(np.arange(len(metadata))).astype(str).str.zfill(6)
config = json.load(open(os.path.join(ARTIFACTS_DIR, "embedding_config.json"), "r"))

print("Embeddings: ", embeddings.shape)
print("Metadata: ", metadata.shape)
print("Configuration: ", config)

Load Model

In [ ]:
model = SentenceTransformer(config["embedding_model"], device = "cuda" if torch.cuda.is_available() else "cpu")
print("Model:", model)
print("Device:", model.device)
print("Embedding dimension:", model.get_embedding_dimension())

Retrieval

In [ ]:
def retrieve(query, top_k = 5):
    query = model.encode(query)

    scores = cosine_similarity(query.reshape(1, -1), embeddings)[0]

    top_k = np.argsort(scores)[::-1][:top_k]

    results = metadata.iloc[top_k].copy()
    results["similarity"] = scores[top_k]

    return results.reset_index(drop = True)

Test retrieval

In [ ]:
query = "What is the chemical class of Allegra?"

results = retrieve(query, top_k = 3)

display(results)

Search

In [ ]:
def search(query, top_k=5):

    results = retrieve(query, top_k = top_k)

    for i, row in results.iterrows():

        print("=" * 80)
        print(f"RESULT #{i + 1}")
        print(f"Similarity: {row['similarity']:.4f}")
        print()

        print(row["document"])

Test search

# Retrieval Evaluation — Before Reranking

This block evaluates the actual MIRx dense retrieval implementation against the ground-truth and test-query files. Each retrieval metric is defined in its own cell so experiments can be inspected and modified independently.

In [ ]:
# ============================================================
# EVALUATION DATA — LOAD AND NORMALIZE
# ============================================================

if GROUND_TRUTH_DIR.exists():
    with open(GROUND_TRUTH_DIR, "r", encoding="utf-8") as f:
        ground_truth = json.load(f)
else:
    ground_truth = {"queries": []}
    print("Retrieval evaluation skipped: no finalized ground_truth.json. Complete human annotation in 005_Ground_Truth_Population_FINAL.ipynb first.")

with open(TEST_QUERIES_DIR, "r", encoding="utf-8") as f:
    test_queries_raw = json.load(f)

# -------------------------
# Ground truth
# -------------------------
if isinstance(ground_truth, dict):
    gt_records_raw = ground_truth.get("queries", ground_truth.get("records", []))
elif isinstance(ground_truth, list):
    gt_records_raw = ground_truth
else:
    raise TypeError("Unsupported ground-truth JSON format.")

gt_records = {
    str(record["query_id"]): record
    for record in gt_records_raw
    if isinstance(record, dict) and "query_id" in record
}

# -------------------------
# Test queries
# -------------------------
if isinstance(test_queries_raw, dict):
    test_queries_raw = (
        test_queries_raw.get("queries")
        or test_queries_raw.get("test_queries")
        or test_queries_raw.get("records")
        or []
    )
elif not isinstance(test_queries_raw, list):
    raise TypeError("Unsupported test-query JSON format.")

def _norm_text(text):
    return " ".join(str(text).lower().split())

# Build query-text -> GT ID lookup as a safe fallback.
gt_query_text_lookup = {}
for qid, record in gt_records.items():
    qtext = record.get("query") or record.get("question") or record.get("query_text")
    if qtext:
        gt_query_text_lookup[_norm_text(qtext)] = qid

normalized_test_queries = []
for i, item in enumerate(test_queries_raw):
    if isinstance(item, str):
        query_text = item
        query_id = gt_query_text_lookup.get(_norm_text(query_text), f"MIRX-Q-{i + 1:04d}")
        metadata_item = {}
    elif isinstance(item, dict):
        query_text = item.get("query") or item.get("question") or item.get("query_text") or item.get("text")
        if not query_text:
            continue
        query_id = item.get("query_id")
        if query_id is None:
            query_id = gt_query_text_lookup.get(_norm_text(query_text), f"MIRX-Q-{i + 1:04d}")
        metadata_item = item
    else:
        continue

    normalized_test_queries.append({
        "query_id": str(query_id),
        "query": str(query_text),
        "metadata": metadata_item
    })

print("=" * 70)
print("EVALUATION DATA")
print("=" * 70)
print(f"Ground-truth records : {len(gt_records)}")
print(f"Test queries         : {len(normalized_test_queries)}")
print(f"GT query IDs         : {list(gt_records)[:5]}")
print(f"Test query IDs       : {[q['query_id'] for q in normalized_test_queries[:5]]}")

matched_ids = [q["query_id"] for q in normalized_test_queries if q["query_id"] in gt_records]
print(f"Matching evaluation queries: {len(matched_ids)}")

if gt_records and not matched_ids:
    raise ValueError(
        "No test-query IDs match the ground truth. "
        "Ensure test_queries.json and MIRx_ground_truth_master.json use the same query_id values "
        "or contain matching query text."
    )


In [ ]:
# ============================================================
# RETRIEVAL RESULT HELPERS
# ============================================================

def get_retrieved_chunk_ids(results):
    if results is None or len(results) == 0:
        return []

    for column in ["chunk_id", "document_id", "id", "chunk", "chunkId", "chunk_identifier"]:
        if column in results.columns:
            return [str(x) for x in results[column].tolist() if pd.notna(x)]

    raise KeyError(
        "No chunk-ID column found in retrieval output. "
        f"Available columns: {list(results.columns)}"
    )

def get_retrieved_document_ids(results):
    if results is None or len(results) == 0:
        return []

    for column in ["document_id", "doc_id", "documentId", "source_id", "source"]:
        if column in results.columns:
            return [str(x) for x in results[column].tolist() if pd.notna(x)]

    return []

print("Metadata columns:")
print(metadata.columns.tolist())


In [ ]:
# ============================================================
# GROUND-TRUTH RELEVANCE HELPERS
# ============================================================

def get_relevant_chunks(gt_record):
    return {str(document_id) for document_id in gt_record.get("relevance_judgments", {})}

def get_relevance_grade(gt_record, chunk_id):
    graded = gt_record.get("relevance_judgments", {})
    return int(graded.get(str(chunk_id), 0))

def get_relevant_documents(gt_record):
    return {
        str(document_id)
        for document_id in gt_record.get("relevance_judgments", {})
    }


In [ ]:
# ============================================================
# 1. Recall@K
# ============================================================

def recall_at_k(retrieved, relevant, k):
    if not relevant:
        return np.nan
    return len(set(retrieved[:k]) & relevant) / len(relevant)


In [ ]:
# ============================================================
# 2. Precision@K
# ============================================================

def precision_at_k(retrieved, relevant, k):
    retrieved_top_k = retrieved[:k]
    if not retrieved_top_k:
        return 0.0
    return sum(chunk_id in relevant for chunk_id in retrieved_top_k) / len(retrieved_top_k)


In [ ]:
# ============================================================
# 3. Mean Reciprocal Rank (MRR)
# ============================================================

def reciprocal_rank(retrieved, relevant):
    for rank, chunk_id in enumerate(retrieved, start=1):
        if chunk_id in relevant:
            return 1.0 / rank
    return 0.0


In [ ]:
# ============================================================
# 4. NDCG@K
# ============================================================

def dcg_at_k(retrieved, gt_record, k):
    score = 0.0
    for rank, chunk_id in enumerate(retrieved[:k], start=1):
        relevance = get_relevance_grade(gt_record, chunk_id)
        score += (2 ** relevance - 1) / math.log2(rank + 1)
    return score

def ndcg_at_k(retrieved, gt_record, k):
    actual_dcg = dcg_at_k(retrieved, gt_record, k)
    relevant = get_relevant_chunks(gt_record)
    ideal_retrieval = sorted(
        relevant,
        key=lambda chunk_id: get_relevance_grade(gt_record, chunk_id),
        reverse=True
    )
    ideal_dcg = dcg_at_k(ideal_retrieval, gt_record, k)
    return 0.0 if ideal_dcg == 0 else actual_dcg / ideal_dcg


In [ ]:
# ============================================================
# 5. Hit Rate@K
# ============================================================

def hit_rate_at_k(retrieved, relevant, k):
    return float(bool(set(retrieved[:k]) & relevant))


In [ ]:
# ============================================================
# 6. Context Precision@K
# ============================================================

def context_precision(retrieved, gt_record, k):
    retrieved_top_k = retrieved[:k]
    if not retrieved_top_k:
        return 0.0
    max_grade = 4
    return float(np.mean([
        get_relevance_grade(gt_record, chunk_id) / max_grade
        for chunk_id in retrieved_top_k
    ]))


In [ ]:
# ============================================================
# 7. Context Recall@K
# ============================================================

def context_recall(retrieved, gt_record, k):
    relevant = get_relevant_chunks(gt_record)
    if not relevant:
        return np.nan
    return len(set(retrieved[:k]) & relevant) / len(relevant)


In [ ]:
# ============================================================
# 8. Retrieval Diversity@K
# ============================================================

def retrieval_diversity(results, k):
    top_results = results.head(k) if results is not None else pd.DataFrame()
    document_ids = get_retrieved_document_ids(top_results)
    if not document_ids:
        return np.nan
    return len(set(document_ids)) / len(document_ids)


In [ ]:
# ============================================================
# RUN RETRIEVAL EVALUATION — DENSE RETRIEVAL / NO RERANKING
# ============================================================

evaluation_rows = []
matched_queries = 0
skipped_queries = 0

for test_item in normalized_test_queries:
    query_id = test_item["query_id"]
    query = test_item["query"]

    if query_id not in gt_records:
        skipped_queries += 1
        continue

    gt_record = gt_records[query_id]
    matched_queries += 1

    retrieval_results = retrieve(
        query,
        top_k=max(TOP_K_VALUES)
    )

    retrieved_chunk_ids = get_retrieved_chunk_ids(retrieval_results)
    relevant_chunk_ids = get_relevant_chunks(gt_record)

    row = {
        "query_id": query_id,
        "query": query,
        "query_type": gt_record.get("query_type", "unknown"),
        "difficulty": gt_record.get("difficulty", "unknown"),
        "num_retrieved": len(retrieved_chunk_ids),
        "num_gold_relevant": len(relevant_chunk_ids)
    }

    for k in TOP_K_VALUES:
        row[f"Recall@{k}"] = recall_at_k(retrieved_chunk_ids, relevant_chunk_ids, k)
        row[f"Precision@{k}"] = precision_at_k(retrieved_chunk_ids, relevant_chunk_ids, k)
        row[f"NDCG@{k}"] = ndcg_at_k(retrieved_chunk_ids, gt_record, k)
        row[f"HitRate@{k}"] = hit_rate_at_k(retrieved_chunk_ids, relevant_chunk_ids, k)
        row[f"ContextPrecision@{k}"] = context_precision(retrieved_chunk_ids, gt_record, k)
        row[f"ContextRecall@{k}"] = context_recall(retrieved_chunk_ids, gt_record, k)
        row[f"RetrievalDiversity@{k}"] = retrieval_diversity(retrieval_results, k)

    row["MRR"] = reciprocal_rank(retrieved_chunk_ids, relevant_chunk_ids)
    evaluation_rows.append(row)

retrieval_metrics_before_reranking = pd.DataFrame(
    evaluation_rows,
    columns=[
        "query_id", "query", "query_type", "difficulty", "num_retrieved",
        "num_gold_relevant", "Recall@1", "Precision@1", "NDCG@1",
        "HitRate@1", "ContextPrecision@1", "ContextRecall@1",
        "RetrievalDiversity@1", "Recall@3", "Precision@3", "NDCG@3",
        "HitRate@3", "ContextPrecision@3", "ContextRecall@3",
        "RetrievalDiversity@3", "Recall@5", "Precision@5", "NDCG@5",
        "HitRate@5", "ContextPrecision@5", "ContextRecall@5",
        "RetrievalDiversity@5", "Recall@10", "Precision@10", "NDCG@10",
        "HitRate@10", "ContextPrecision@10", "ContextRecall@10",
        "RetrievalDiversity@10", "MRR",
    ],
)

print("=" * 70)
print("RETRIEVAL EVALUATION COMPLETE")
print("=" * 70)
print(f"Test queries       : {len(normalized_test_queries)}")
print(f"Matched GT queries : {matched_queries}")
print(f"Skipped queries    : {skipped_queries}")
print(f"Evaluated queries  : {len(retrieval_metrics_before_reranking)}")

if retrieval_metrics_before_reranking.empty:
    print("No retrieval metrics were produced because finalized ground truth is unavailable.")

display(retrieval_metrics_before_reranking.head())


In [ ]:
# ============================================================
# RETRIEVAL METRICS — AGGREGATE RESULTS
# ============================================================

metric_summary = {}

for k in TOP_K_VALUES:
    for metric_name in [
        "Recall", "Precision", "NDCG", "HitRate",
        "ContextPrecision", "ContextRecall", "RetrievalDiversity"
    ]:
        column = f"{metric_name}@{k}"
        metric_summary[column] = retrieval_metrics_before_reranking[column].mean()

metric_summary["MRR"] = retrieval_metrics_before_reranking["MRR"].mean()

retrieval_summary_df = pd.DataFrame(
    metric_summary.items(),
    columns=["Metric", "Score"]
)
retrieval_summary_df["Score"] = retrieval_summary_df["Score"].round(4)

print("=" * 70)
print("MIRx — BASELINE RETRIEVAL RESULTS")
print("Dense Retrieval | No Reranking")
print("=" * 70)
display(retrieval_summary_df)
